# 02 - Camada Bronze: landing-zone para Delta Lake

Le os arquivos CSV da landing-zone com Apache Spark e salva no bucket bronze no formato Delta Lake.

In [ ]:
import os
from dotenv import load_dotenv
from pyspark.sql import SparkSession

load_dotenv()

MINIO_ENDPOINT  = os.getenv("MINIO_ENDPOINT")
MINIO_ACCESS_KEY = os.getenv("MINIO_ACCESS_KEY")
MINIO_SECRET_KEY = os.getenv("MINIO_SECRET_KEY")
LANDING          = os.getenv("MINIO_LANDING_BUCKET")
BRONZE           = os.getenv("MINIO_BRONZE_BUCKET")

## Criacao da SparkSession

Todos os JARs (delta-spark, hadoop-aws, aws-java-sdk) estao em pyspark/jars/ e sao carregados pelo JVM antes da inicializacao das extensoes Delta.

In [ ]:
spark = (
    SparkSession.builder
    .appName("landing-to-bronze")
    .config("spark.sql.extensions",
            "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog",
            "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .config("spark.hadoop.fs.s3a.endpoint",          MINIO_ENDPOINT)
    .config("spark.hadoop.fs.s3a.access.key",        MINIO_ACCESS_KEY)
    .config("spark.hadoop.fs.s3a.secret.key",        MINIO_SECRET_KEY)
    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    .config("spark.hadoop.fs.s3a.impl",
            "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")
    .config("spark.driver.memory", "2g")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")
print("Spark versao:", spark.version)
print("extensions:",   spark.conf.get("spark.sql.extensions", "NAO CONFIGURADO"))
print("catalog:",      spark.conf.get("spark.sql.catalog.spark_catalog", "NAO CONFIGURADO"))

## Garantir que o bucket `bronze` existe

O notebook 01 criou o `landing-zone`. Aqui criamos o `bronze` se ainda não existir.

In [ ]:
import boto3
from botocore.client import Config

_s3 = boto3.client(
    "s3",
    endpoint_url=MINIO_ENDPOINT,
    aws_access_key_id=MINIO_ACCESS_KEY,
    aws_secret_access_key=MINIO_SECRET_KEY,
    config=Config(signature_version="s3v4"),
)

_existing = [b["Name"] for b in _s3.list_buckets()["Buckets"]]
if BRONZE not in _existing:
    _s3.create_bucket(Bucket=BRONZE)
    print(f"Bucket '{BRONZE}' criado.")
else:
    print(f"Bucket '{BRONZE}' já existe.")

## Leitura e escrita em Delta Lake

In [ ]:
TABLES = [
    "apolice", "carro", "cliente", "endereco",
    "estado", "marca", "modelo", "municipio",
    "regiao", "sinistro", "telefone"
]

for col in TABLES:
    src  = f"s3a://{LANDING}/{col}/{col}.csv"
    dest = f"s3a://{BRONZE}/{col}"

    df = spark.read.option("header", "true").option("inferSchema", "true").csv(src)

    (
        df.write
        .format("delta")
        .mode("overwrite")
        .save(dest)
    )

    count = spark.read.format("delta").load(dest).count()
    print(f"  {col}: {count} registros")

print("\nCamada bronze concluida.")

## Verificacao dos schemas

In [ ]:
for col in TABLES:
    dest = f"s3a://{BRONZE}/{col}"
    df   = spark.read.format("delta").load(dest)
    print(f"\n=== {col} ===")
    df.printSchema()
    df.show(3, truncate=True)

## Historico Delta (versoes da tabela)

In [ ]:
from delta.tables import DeltaTable

for col in TABLES:
    dest    = f"s3a://{BRONZE}/{col}"
    dt      = DeltaTable.forPath(spark, dest)
    history = dt.history(1).select("version", "timestamp", "operation")
    print(f"{col}:")
    history.show(truncate=False)